# 1- Transformer Encoder–Decoder dengan PyTorch

Dibuat oleh <b>NICHOLAS DOMINIC</b>
<br>- ReWork Academy, 2026 -

Notebook ini mengimplementasikan Transformer **secara manual** tanpa library `transformers` / Hugging Face.
<br>Komponen yang dibangun meliputi:

- scaled dot-product self-attention dan multi-head attention (proyeksi Q, K, V);
- positional encoding, residual connection, dan layer normalization;
- encoder dan decoder, termasuk causal mask dan cross-attention;
- training, validation, prediksi autoregresif, dan visualisasi bobot attention.

Dataset dibuat secara sintetis untuk tugas membalik urutan angka, jadi tidak perlu mengunduh dataset.
<br>Contoh: `[4, 1, 7, 2] → [2, 7, 1, 4]`.

**Cara pakai:**
<br>Jalankan sel dari atas ke bawah.

Dependensi: `torch` dan `matplotlib`
<br>Jika belum, jalankan `pip install torch matplotlib`

## 1. Import dan konfigurasi

In [3]:
!pip install torch

In [4]:
import math
import random
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset

In [5]:
# ID token khusus dan angka 0..15.
PAD_ID = 0
BOS_ID = 1  # Beginning of sequence
EOS_ID = 2  # End of sequence
NUM_SYMBOLS = 16
VOCAB_SIZE = NUM_SYMBOLS + 3

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# TODO: Set opsi device sebagai akselerator
if torch.cuda.is_available():
  DEVICE = torch.device("cuda") #NVIDIA GPU Series
# elif hasattr(torch.backends, mps) and torch.backend.mps.is_available():
#   DEVICE = torch.device("mps")
else:
  DEVICE =  torch.device("cpu")

: 

In [ ]:
# Untuk demo kecil, satu thread CPU biasanya mengurangi overhead
if DEVICE.type == "cpu":
    torch.set_num_threads(1)

OUTPUT_DIR = Path("transformer_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"Grafik akan disimpan di: {OUTPUT_DIR.resolve()}")

## 2. Positional encoding dan multi-head attention

Untuk setiap head, attention dihitung dengan rumus:

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^\top}{\sqrt{d_k}}
\right)V
$$

Mask padding mencegah model memperhatikan token kosong.
<br>Causal mask di decoder mencegah token melihat posisi masa depan.

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding, dibuat manual tanpa library Transformer."""

    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        positions = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        scale = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(positions * scale)
        
        # Irisan cos juga aman untuk d_model genap; konfigurasi model menggunakan 64.
        pe[:, 1::2] = torch.cos(positions * scale[: pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head attention manual: Q/K/V, scaled dot-product, masking, output projection."""

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model harus habis dibagi n_heads")
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        # (batch, sequence, d_model) -> (batch, heads, sequence, head_dim)
        batch, seq_len, _ = x.shape
        return x.view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        key_padding_mask: torch.Tensor | None = None,
        attn_mask: torch.Tensor | None = None,
        return_attention: bool = False,
    ):
        q = self._split_heads(self.q_proj(query))
        k = self._split_heads(self.k_proj(key))
        v = self._split_heads(self.v_proj(value))

        # Scaled dot-product attention: softmax(Q K^T / sqrt(d_k)) V
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # True pada mask berarti posisi key tidak boleh dilihat.
        if key_padding_mask is not None:
            scores = scores.masked_fill(
                key_padding_mask[:, None, None, :], torch.finfo(scores.dtype).min
            )
        # Causal mask: decoder tidak boleh melihat token masa depan.
        if attn_mask is not None:
            scores = scores.masked_fill(
                attn_mask[None, None, :, :], torch.finfo(scores.dtype).min
            )

        attention_weights = torch.softmax(scores, dim=-1)
        attended = torch.matmul(self.attn_dropout(attention_weights), v)
        attended = attended.transpose(1, 2).contiguous()
        batch, query_len, _, _ = attended.shape
        attended = attended.view(batch, query_len, self.d_model)
        output = self.out_proj(attended)

        if return_attention:
            # Bentuk weights: (batch, heads, panjang_query, panjang_key)
            return output, attention_weights
        return output, None

## 3. Feed-forward network, encoder layer, dan decoder layer

In [ ]:
class FeedForward(nn.Module):
    """Feed-forward network kecil yang diterapkan pada setiap posisi token."""

    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_padding_mask, return_attention=False):
        attention_out, weights = self.self_attention(
            x,
            x,
            x,
            key_padding_mask=src_padding_mask,
            return_attention=return_attention,
        )
        x = self.norm1(x + self.dropout(attention_out))
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x, weights

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x,
        memory,
        tgt_padding_mask,
        src_padding_mask,
        causal_mask,
        return_attention=False,
    ):
        self_out, self_weights = self.self_attention(
            x,
            x,
            x,
            key_padding_mask=tgt_padding_mask,
            attn_mask=causal_mask,
            return_attention=return_attention,
        )
        x = self.norm1(x + self.dropout(self_out))

        cross_out, cross_weights = self.cross_attention(
            x,
            memory,
            memory,
            key_padding_mask=src_padding_mask,
            return_attention=return_attention,
        )
        x = self.norm2(x + self.dropout(cross_out))
        x = self.norm3(x + self.dropout(self.feed_forward(x)))
        return x, self_weights, cross_weights

## 4. Susun Transformer encoder–decoder

In [ ]:
class Transformer(nn.Module):
    """Encoder-decoder Transformer; semua blok attention ditulis manual."""

    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        d_model=64,
        n_heads=4,
        n_layers=2,
        d_ff=128,
        dropout=0.1,
        max_len=128,
    ):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.position = PositionalEncoding(d_model, dropout, max_len=max_len)
        
        self.encoder_layers = nn.ModuleList(
            # TODO: Lengkapi encoder layer
        )
        
        self.decoder_layers = nn.ModuleList(
            # TODO: Lengkapi decoder layer
        )
        
        self.output_projection = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt_in, return_attention=False):
        src_padding_mask = src.eq(PAD_ID)  # (batch, src_len)
        tgt_padding_mask = tgt_in.eq(PAD_ID)  # (batch, tgt_len)
        causal_mask = torch.triu(
            torch.ones(
                tgt_in.size(1), tgt_in.size(1), dtype=torch.bool, device=tgt_in.device
            ),
            diagonal=1,
        )

        src_x = self.position(self.embedding(src) * math.sqrt(self.d_model))
        tgt_x = self.position(self.embedding(tgt_in) * math.sqrt(self.d_model))

        encoder_weights = None
        for layer in self.encoder_layers:
            src_x, encoder_weights = layer(
                src_x, src_padding_mask, return_attention=return_attention
            )
        memory = src_x

        decoder_self_weights = None
        cross_weights = None
        for layer in self.decoder_layers:
            tgt_x, decoder_self_weights, cross_weights = layer(
                tgt_x,
                memory,
                tgt_padding_mask,
                src_padding_mask,
                causal_mask,
                return_attention=return_attention,
            )

        logits = self.output_projection(tgt_x)
        attention = None
        if return_attention:
            # Bobot attention dari layer terakhir, untuk inspeksi/visualisasi.
            attention = {
                "encoder_self": encoder_weights,
                "decoder_self": decoder_self_weights,
                "cross": cross_weights,
            }
        return logits, attention

## 5. Dataset sintetis, training utilities, dan decoding

Setiap input berisi angka token acak. Target dimulai dengan `BOS`, dilanjutkan angka dalam urutan terbalik, lalu diakhiri `EOS`. Padding membuat panjang sampel dalam batch seragam.

In [ ]:
def make_dataset(sample_count: int, max_seq_len: int, seed: int) -> TensorDataset:
    """Membuat pasangan input dan target sintesis dengan padding."""
    generator = torch.Generator().manual_seed(seed)
    src_all = torch.full((sample_count, max_seq_len), PAD_ID, dtype=torch.long)
    tgt_all = torch.full((sample_count, max_seq_len + 2), PAD_ID, dtype=torch.long)

    for i in range(sample_count):
        length = int(torch.randint(3, max_seq_len + 1, (1,), generator=generator).item())
        sequence = torch.randint(
            3, VOCAB_SIZE, (length,), generator=generator, dtype=torch.long
        )
        src_all[i, :length] = sequence
        reversed_sequence = sequence.flip(0)
        tgt_all[i, 0] = BOS_ID
        tgt_all[i, 1 : length + 1] = reversed_sequence
        tgt_all[i, length + 1] = EOS_ID

    return TensorDataset(src_all, tgt_all)

In [ ]:
def run_epoch(model, loader, criterion, device, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_batches = 0

    for src, tgt in loader:
        src = src.to(device)
        tgt = tgt.to(device)
        tgt_in = tgt[:, :-1]
        expected = tgt[:, 1:]

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits, _ = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), expected.reshape(-1))
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(total_batches, 1)

In [ ]:
def decode_token_ids(token_ids):
    """Mengubah ID token model kembali menjadi angka yang mudah dibaca."""
    result = []
    for token_id in token_ids:
        if token_id == EOS_ID:
            break
        if token_id >= 3:
            result.append(token_id - 3)
    return result

In [ ]:
@torch.no_grad()
def greedy_decode(model, src, max_seq_len, device):
    """Inferensi autoregresif sederhana: prediksi token satu per satu."""
    model.eval()
    src_batch = src.unsqueeze(0).to(device)
    generated = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)

    for _ in range(max_seq_len + 1):
        logits, _ = model(src_batch, generated)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
        if int(next_token.item()) == EOS_ID:
            break
    return generated[0, 1:].tolist()

## 6. Fungsi visualisasi

Grafik pertama membandingkan loss training dan validation per epoch. Grafik kedua memperlihatkan bobot attention untuk satu head pada layer terakhir: encoder self-attention, decoder causal self-attention, dan decoder cross-attention.

In [ ]:
def plot_training_curves(train_losses, val_losses, output_dir, show_plot=True):
    epochs = range(1, len(train_losses) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, train_losses, marker="o", label="Training loss")
    ax.plot(epochs, val_losses, marker="o", label="Validation loss")
    ax.set_title("Training vs Validation Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-entropy loss")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    path = output_dir / "training_validation_loss.png"
    fig.savefig(path, dpi=160)
    print(f"Grafik loss disimpan: {path.resolve()}")
    if show_plot:
        plt.show()
    plt.close(fig)

In [ ]:
def plot_attention(model, dataset, max_seq_len, device, output_dir, show_plot=True):
    """Plot satu head dari encoder self-attention, decoder self-attention, cross-attention."""
    src, tgt = dataset[0]
    source_tokens = src[src.ne(PAD_ID)].tolist()
    decoder_input = tgt[:-1].unsqueeze(0).to(device)  # teacher forcing untuk visualisasi
    src_batch = src.unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        _, attention = model(src_batch, decoder_input, return_attention=True)

    src_len = len(source_tokens)
    # Decoder input yang relevan: BOS + token target terbalik.
    tgt_len = int(tgt.ne(PAD_ID).sum().item()) - 1
    labels_src = [str(t - 3) for t in source_tokens]
    labels_tgt = ["BOS"] + [str(t - 3) for t in tgt[1 : tgt_len].tolist()]

    matrices = [
        attention["encoder_self"][0, 0, :src_len, :src_len].cpu().numpy(),
        attention["decoder_self"][0, 0, :tgt_len, :tgt_len].cpu().numpy(),
        attention["cross"][0, 0, :tgt_len, :src_len].cpu().numpy(),
    ]
    titles = [
        "Encoder self-attention (head 1)",
        "Decoder causal self-attention (head 1)",
        "Decoder cross-attention (head 1)",
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for i, ax in enumerate(axes):
        image = ax.imshow(matrices[i], cmap="viridis", aspect="auto")
        ax.set_title(titles[i])
        ax.set_xlabel("Key / token yang dilihat")
        ax.set_ylabel("Query / token yang memerhatikan")
        ax.set_xticks(range(len(labels_src if i != 1 else labels_tgt)))
        ax.set_xticklabels(labels_src if i != 1 else labels_tgt)
        row_labels = labels_src if i == 0 else labels_tgt
        ax.set_yticks(range(len(row_labels)))
        ax.set_yticklabels(row_labels)
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle("Contoh bobot attention — layer terakhir, head pertama")
    fig.tight_layout()
    path = output_dir / "attention_maps.png"
    fig.savefig(path, dpi=160)
    print(f"Grafik attention disimpan: {path.resolve()}")
    if show_plot:
        plt.show()
    plt.close(fig)

## 7. Latih model dan evaluasi

Ubah nilai `EPOCHS`, `TRAIN_SIZE`, atau parameter lain di sel ini untuk mencoba konfigurasi berbeda. Konfigurasi awal dibuat relatif ringan untuk demonstrasi.

In [ ]:
# Hyperparameter demo — silakan ubah untuk bereksperimen
EPOCHS = 10
BATCH_SIZE = 64
TRAIN_SIZE = 1000
VAL_SIZE = 200
MAX_SEQ_LEN = 8

In [ ]:
train_dataset = make_dataset(TRAIN_SIZE, MAX_SEQ_LEN, SEED)
val_dataset = make_dataset(VAL_SIZE, MAX_SEQ_LEN, SEED + 1)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model = Transformer(max_len=MAX_SEQ_LEN + 8).to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
print("=== Demonstrasi Transformer manual ===")
print(f"Train: {len(train_dataset)} sampel | Validation: {len(val_dataset)} sampel")
print("Arsitektur: encoder-decoder | 2 layer | 4 heads | d_model=64 | d_ff=128")
print("Tugas: [a, b, c] -> [c, b, a]")
print()

train_losses, val_losses = [], []
for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, criterion, DEVICE, optimizer)
    val_loss = run_epoch(model, val_loader, criterion, DEVICE)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"training loss: {train_loss:.4f} | validation loss: {val_loss:.4f}"
    )

print("\n=== Contoh prediksi autoregresif ===")
for idx in range(min(5, len(val_dataset))):
    src, _ = val_dataset[idx]
    actual_source = decode_token_ids(src.tolist())
    expected = list(reversed(actual_source))
    predicted = decode_token_ids(greedy_decode(model, src, MAX_SEQ_LEN, DEVICE))
    print(f"Input: {actual_source} | Target: {expected} | Prediksi: {predicted}")

In [ ]:
# Periksa ukuran tensor attention dari forward pass.
sample_src, sample_tgt = val_dataset[0]
with torch.no_grad():
    _, attn = model(
        sample_src.unsqueeze(0).to(DEVICE),
        sample_tgt[:-1].unsqueeze(0).to(DEVICE),
        return_attention=True,
    )
print("\n=== Bentuk tensor attention (batch, heads, query_len, key_len) ===")
print(f"Encoder self-attention : {tuple(attn['encoder_self'].shape)}")
print(f"Decoder self-attention : {tuple(attn['decoder_self'].shape)}")
print(f"Decoder cross-attention: {tuple(attn['cross'].shape)}")

In [ ]:
# TODO: Coba plot hasil training Transformer
plot_training_curves(...)

In [ ]:
# TODO: Coba plot hasil attention
plot_attention(...)